**Install Required Packages**

In [1]:
!pip install google-cloud-bigquery google-cloud-storage google-auth


**Import libraries and initialize client instances**


In [2]:
import os
import asyncio
import google.auth

os.environ['GOOGLE_CLOUD_PROJECT']

GOOGLE_CLOUD_PROJECT = "us-gcp-ame-con-e5556-npd-1"
GOOGLE_CLOUD_LOCATION = "us-central1"
MODEL_NAME = "gemini-2.5-flash"

os.environ['GOOGLE_CLOUD_PROJECT'] = GOOGLE_CLOUD_PROJECT
os.environ['GOOGLE_CLOUD_LOCATION'] = GOOGLE_CLOUD_LOCATION
os.environ['MODEL_NAME'] = MODEL_NAME

google.auth.default()

(<google.auth.compute_engine.credentials.Credentials at 0x794403590ce0>,
 'us-gcp-ame-con-e5556-npd-1')

In [3]:
!gcloud auth application-default login


You are running on a Google Compute Engine virtual machine.
The service credentials associated with this virtual machine
will automatically be used by Application Default
Credentials, so it is not necessary to use this command.

If you decide to proceed anyway, your user credentials may be visible
to others with access to this virtual machine. Are you sure you want
to authenticate with your personal account?

Do you want to continue (Y/n)?  

Command killed by keyboard interrupt

^C


In [4]:
os.getenv("GOOGLE_CLOUD_PROJECT")
os.getenv("GOOGLE_CLOUD_LOCATION")

'us-central1'

In [5]:
from google.cloud import bigquery
from google.cloud import storage

# Initialize BigQuery and Storage clients
bigquery_client = bigquery.Client()
storage_client = storage.Client()

# --- Configuration for your data load ---
# Replace with your project ID, dataset ID, table ID, and GCS bucket details
project_id = "us-gcp-ame-con-e5556-npd-1"
location = "us-central1"
dataset_id = "emergency_response"
raw_table_id = "emergency_response_data_raw"
gold_table_id = "emergency_response_training_data"
model_id = "emergency_response_model"
source_bucket_name = "gs://labs.roitraining.com/data-to-ai-workshop"
source_file_name = "gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv"

**Create Dataset**

In [6]:
sql_ddl_dataset_statement = f"""
CREATE SCHEMA IF NOT EXISTS `{project_id}.{dataset_id}`
OPTIONS(
    location = '{location}',
    default_table_expiration_days=14
);
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_dataset_statement)

# Wait for the job to complete
query_job.result()

print(f"Dataset '{dataset_id}' created or already exists in project '{project_id}' at location '{location}'.")


Dataset 'emergency_response' created or already exists in project 'us-gcp-ame-con-e5556-npd-1' at location 'us-central1'.


**Create Table**

In [7]:
sql_ddl_statement = f"""
DROP TABLE IF EXISTS `{project_id}.{dataset_id}.{raw_table_id}`;

CREATE TABLE `{project_id}.{dataset_id}.{raw_table_id}` (
    call_id STRING,
    call_timestamp TIMESTAMP,
    call_type STRING,
    location STRING,
    weather_conditions STRING,
    day_of_week STRING,
    time_of_day INTEGER,
    traffic_level STRING,
    distance_to_station NUMERIC,
    units_available INTEGER,
    response_time FLOAT64
    );
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_statement)

# Wait for the job to complete
query_job.result()

print(f"Table '{raw_table_id}' created or replaced in dataset '{dataset_id}'.")


Table 'emergency_response_data_raw' created or replaced in dataset 'emergency_response'.


**Load data into Big Query Table**

In [8]:
sql_ddl_statement = f"""
LOAD DATA INTO `{project_id}.{dataset_id}.{raw_table_id}`
(
    call_id STRING,
    call_timestamp TIMESTAMP,
    call_type STRING,
    location STRING,
    weather_conditions STRING,
    day_of_week STRING,
    time_of_day INTEGER,
    traffic_level STRING,
    distance_to_station NUMERIC,
    units_available INTEGER,
    response_time FLOAT64
)
FROM FILES (
format = 'CSV',
field_delimiter = ',',
max_bad_records = 10,
uris = ['{source_file_name}']);
"""
# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_statement)

# Wait for the job to complete
query_job.result()

print(f"Table '{raw_table_id}' loaded with raw data from '{source_file_name}'.")

Table 'emergency_response_data_raw' loaded with raw data from 'gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv'.


In [9]:
df = bigquery_client.list_rows(f"{project_id}.{dataset_id}.{raw_table_id}").to_dataframe()
print(df.head())


  call_id            call_timestamp call_type    location weather_conditions  \
0   35957 2023-01-01 00:05:53+00:00      Fire    Highland              Rainy   
1   20832 2023-01-01 00:20:47+00:00      Fire     Oakmont              Rainy   
2   27949 2023-01-01 00:33:27+00:00      Fire   Riverside              Windy   
3   20199 2023-01-01 00:48:29+00:00      Fire   Riverside              Windy   
4   46938 2023-01-01 00:50:44+00:00    Rescue  Brookfield              Sunny   

  day_of_week  time_of_day traffic_level distance_to_station  units_available  \
0      Sunday            0          High        21.450000000                3   
1      Sunday            0          High        22.290000000                6   
2      Sunday            0          High        17.190000000               14   
3      Sunday            0          High        17.390000000               14   
4      Sunday            0          High        22.500000000               14   

   response_time  
0          23

**Create ML Model with Auto Split of training data**

In [22]:
create_model_statement = f"""
CREATE OR REPLACE MODEL `{dataset_id}.{model_id}`
OPTIONS (model_type='linear_reg',
input_label_cols=['response_time'],
data_split_method = 'AUTO_SPLIT'
) AS
SELECT * EXCEPT(call_id, call_timestamp)
FROM `{project_id}.{dataset_id}.{raw_table_id}`
"""
# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(create_model_statement)

# Wait for the job to complete
query_job.result()
print(f"Model `{dataset_id}.{model_id}` created successfully")

Model `emergency_response.emergency_response_model` created successfully


**Evaluate the Model**

In [23]:
eval_model_statement = f"""
SELECT *
FROM ML.EVALUATE(MODEL `{dataset_id}.{model_id}`, (
Select * from `{project_id}.{dataset_id}.{raw_table_id}`
where call_timestamp between '2023-07-01' and '2023-07-31'));
"""
# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(eval_model_statement)

# Wait for the job to complete
query_job.result()
print(f"Model `{dataset_id}.{model_id}` evaluated successfully")

Model `emergency_response.emergency_response_model` evaluated successfully


In [24]:
eval_df=query_job.to_dataframe()
print(eval_df)

   mean_absolute_error  mean_squared_error  mean_squared_log_error  \
0             1.743458            4.740304                0.014649   

   median_absolute_error  r2_score  explained_variance  
0               1.481277  0.830749            0.830808  


**Predict Emergency Response Time**

In [27]:
# prompt: generate sql statement to predict the response time using the model in variable `{dataset_id}.{model_id}`

predict_statement = f"""
SELECT *
FROM
  ML.PREDICT(MODEL `{dataset_id}.{model_id}`, (
    SELECT
      'Police' as call_type,
      'Oakmont' as location,
      'Sunny' as weather_conditions,
      'Tuesday' as day_of_week,
      CAST('8' as INT64) as time_of_day,
      'High' as traffic_level,
      CAST('10.67' as NUMERIC) as distance_to_station,
      CAST('10' as INT64) as units_available,

      UNION ALL
      SELECT
      'Fire' as call_type,
      'Greenfield' as location,
      'Rainy' as weather_conditions,
      'Saturday' as day_of_week,
      CAST('12' as INT64) as time_of_day,
      'Medium' as traffic_level,
      CAST('5.1' as NUMERIC) as distance_to_station,
      CAST('5' as INT64) as units_available,

      UNION ALL
      SELECT
      'Medical' as call_type,
      'Brookfield' as location ,
      'Cloudy' as weather_conditions,
      'Monday' as day_of_day,
      CAST('16' as INT64) as time_of_day,
      'Low' as traffic_level,
      CAST('24' as NUMERIC) as distance_to_station,
      CAST('2' as INT64) as units_available,

  ))
"""
# Execute the DDL statement using the BigQuery client
predicted_df = bigquery_client.query(predict_statement).to_dataframe()

print(predicted_df)


   predicted_response_time call_type    location weather_conditions  \
0                16.134988    Police     Oakmont              Sunny   
1                12.284987      Fire  Greenfield              Rainy   
2                19.717523   Medical  Brookfield             Cloudy   

  day_of_week  time_of_day traffic_level distance_to_station  units_available  
0     Tuesday            8          High        10.670000000               10  
1    Saturday           12        Medium         5.100000000                5  
2      Monday           16           Low        24.000000000                2  
